[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# A Product Catalog


## What you will be able to do

Build one real thing with everything this guide taught: a catalog of two hundred thousand products,
loaded with PyMongo because loading is what PyMongo is for, indexed for the queries it actually
serves and measured rather than guessed at, and read through Beanie models so the code that uses it
gets typed objects. And say, for each half, why it got the tool it got.


## The idea

### The problem

Every notebook so far has taught one thing against a collection made for it. A real catalog needs
all of them at once, and the interesting part is not any single technique: it is that the same
collection is written by one library and read by another, and that the index you need depends on the
query you serve rather than on the shape of the document.

### What this program is

Three pieces. A loader, which takes a list of dictionaries and puts them in as fast as MongoDB
will take them. An index step, which creates what the queries need and proves it with `explain`. And
a model, which the application code sees instead of dictionaries.

### Why both libraries

PyMongo loads: `insert_many` over plain dictionaries has nothing to validate and nothing to
construct, which is exactly what you want for two hundred thousand rows. Beanie reads: the code that
serves a request wants `product.price`, not `document["price"]`, and wants to be told when a
document does not fit.

They are the same collection. Nothing converts between them, because BSON is what both of them
speak.

### Where this shows up

Any system with an import on one side and an application on the other, which is most of them. The
mistake it prevents is loading two hundred thousand documents through a model that validates each
one, which is slow for no benefit on data you generated yourself.

### What this notebook covers

The load. The indexes, chosen from the queries and measured. The model over the same collection. A
report through the aggregation pipeline. Then the four failures that only appear once the pieces are
together.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio

import pymongo
from beanie import Document, init_beanie
from pymongo import AsyncMongoClient


class Product(Document):
    sku: str
    kind: str
    price: float

    class Settings:
        name = "catalog_demo"


async def main():
    shop = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop").get_default_database()
    shop.catalog_demo.drop()

    rows = [{"sku": f"S-{n:06d}", "kind": ["tool", "part"][n % 2], "price": float(n)}
            for n in range(50_000)]
    shop.catalog_demo.insert_many(rows)                # PyMongo: the load

    client = AsyncMongoClient("mongodb://127.0.0.1:27017/shop")
    await init_beanie(database=client.get_default_database(), document_models=[Product])

    found = await Product.find(Product.kind == "tool", Product.price < 10).to_list()

    print("loaded:", shop.catalog_demo.count_documents({}), "documents")
    print("read back as models:", [p.sku for p in found])
    print("one tool costs", found[0].price + 0.5, "with a bit added, because price is a float")
    print("loaded with PyMongo, read with Beanie, and both saw the same collection")
    await client.close()


asyncio.run(main())
```

```
loaded: 50000 documents
read back as models: ['S-000000', 'S-000002', 'S-000004', 'S-000006', 'S-000008']
one tool costs 0.5 with a bit added, because price is a float
loaded with PyMongo, read with Beanie, and both saw the same collection
```

Fifty thousand dictionaries in, five typed objects out. Nothing translated between the two halves,
because there is nothing to translate: a collection is a collection, and the model is a way of
looking at it.


## Setup

Eleven imports, MongoDB, the boot cell, and three helpers.

- `pymongo` loads and measures, `beanie` reads, `pydantic` declares the report's shape
- `IndexModel` declares indexes on a model and `In` is one query operator the catalog needs
- `subprocess`, `os`, `sys`, `time`, `random`, `version` and `PackageNotFoundError` run the boot cell

`catalog_rows` builds two hundred thousand dictionaries from `random.seed(1)`, so every run of this
notebook loads exactly the same catalog. `plan` runs `explain` and pulls out what matters. `CATALOG`
is the collection name, in one place.

This notebook leaves the guide's own `products` collection alone and builds its own.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pydantic
import pymongo
from beanie import Document, init_beanie
from beanie.operators import In
from pymongo import AsyncMongoClient, IndexModel

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

CATALOG = "catalog"                                                 # one place, spelled once
SIZE = 200_000                                                      # big enough for explain to matter


def failed(error):
    """A failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def plan(cursor):
    """How the server answered, which is the only honest way to judge an index."""
    explained = cursor.explain()
    winner = explained["queryPlanner"]["winningPlan"]
    stats = explained["executionStats"]
    return {"stage": winner.get("inputStage", winner).get("stage"),
            "documents": stats["totalDocsExamined"],
            "returned": stats["nReturned"]}


def catalog_rows(size=None):
    """The rows to load, built once from a fixed seed so every run of this notebook agrees."""
    size = SIZE if size is None else size
    random.seed(1)
    kinds = ["laptop", "monitor", "keyboard", "mouse", "cable"]
    makers = ["Aster", "Belden", "Corvid", "Dalgo"]
    return [{"sku": f"SKU-{number:07d}",
             "name": f"{makers[number % 4]} {kinds[number % 5]} {number}",
             "maker": makers[number % 4],
             "kind": kinds[number % 5],
             "price": round(random.uniform(5, 2000), 2),
             "stock": random.randint(0, 500)}
            for number in range(size)]


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products (the guide's own, untouched by this notebook)")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  200000 products (the guide's own, untouched by this notebook)
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 200000


## Worked examples

### The load

Plain dictionaries, in batches, with PyMongo:


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop[CATALOG].drop()

rows = catalog_rows()
start = time.perf_counter()
for first in range(0, len(rows), 10_000):
    shop[CATALOG].insert_many(rows[first:first + 10_000])
took = time.perf_counter() - start

print("loaded:", shop[CATALOG].count_documents({}), "documents")
print("in batches of ten thousand, which keeps each request well under the 16MB limit")


loaded: 200000 documents
in batches of ten thousand, which keeps each request well under the 16MB limit


The timing is not printed because on this machine it means nothing; what matters is the shape.
`insert_many` in batches is the fastest way in that does not involve a file, and the batch size is
about the request rather than the documents: a batch has to fit in one message.

Beanie could have done this. It would have constructed two hundred thousand model instances and
validated every one, which is work you want when the data comes from outside and waste when you
generated it yourself.

### What the catalog is for

The indexes come from the queries, so the queries come first:


In [3]:
QUERIES = {
    "one sku":          {"sku": "SKU-0000123"},
    "a kind":           {"kind": "laptop"},
    "a kind and price": {"kind": "laptop", "price": {"$lt": 100}},
    "a maker in stock": {"maker": "Aster", "stock": {"$gt": 0}},
}

print("with only the _id index:")
for label, query in QUERIES.items():
    measured = plan(shop[CATALOG].find(query))
    print(f"  {label:18} {measured['stage']:9} read {measured['documents']:6} "
          f"returned {measured['returned']:6}")


with only the _id index:
  one sku            COLLSCAN  read 200000 returned      1
  a kind             COLLSCAN  read 200000 returned  40000
  a kind and price   COLLSCAN  read 200000 returned   1895
  a maker in stock   COLLSCAN  read 200000 returned  49907


Four collection scans, and the first one is the worst: two hundred thousand documents read to return
a single product.

### The indexes those queries need


In [4]:
shop[CATALOG].create_index("sku", name="sku_unique", unique=True)
shop[CATALOG].create_index([("kind", 1), ("price", 1)], name="kind_price")
shop[CATALOG].create_index([("maker", 1), ("stock", 1)], name="maker_stock")

print("with three indexes:")
for label, query in QUERIES.items():
    measured = plan(shop[CATALOG].find(query))
    waste = measured["documents"] / measured["returned"] if measured["returned"] else 0
    print(f"  {label:18} {measured['stage']:14} read {measured['documents']:6} "
          f"returned {measured['returned']:6}  waste {waste:.1f}")


with three indexes:
  one sku            EXPRESS_IXSCAN read      1 returned      1  waste 1.0
  a kind             IXSCAN         read  40000 returned  40000  waste 1.0
  a kind and price   IXSCAN         read   1895 returned   1895  waste 1.0
  a maker in stock   IXSCAN         read  49907 returned  49907  waste 1.0


Every one of them now reads what it returns. The `sku` index is unique as well, so it is both the
fast path and the constraint that keeps the catalog's key unique, which is two jobs from one index.

Note that `kind_price` serves both the kind query and the kind-and-price query, because of the
prefix rule from **Indexes**. Three indexes cover four queries.

### The model over the same collection


In [5]:
class Product(Document):
    sku: str
    name: str
    maker: str
    kind: str
    price: float
    stock: int = 0

    class Settings:
        name = CATALOG
        indexes = [IndexModel([("sku", 1)], name="sku_unique", unique=True)]

    def is_available(self):
        return self.stock > 0


async_client = AsyncMongoClient(URI)
await init_beanie(database=async_client.get_default_database(), document_models=[Product])

one = await Product.find_one(Product.sku == "SKU-0000123")
print("as a model:", one.name, "|", one.price, "| available:", one.is_available())
print("and the collection it read is the one PyMongo wrote:", Product.get_pymongo_collection().name)


as a model: Dalgo mouse 123 | 347.18 | available: True
and the collection it read is the one PyMongo wrote: catalog


`Settings.indexes` declares the same unique index the loader created. `init_beanie` asks for it
again, which does nothing because it already exists, and that is what makes it safe to have in both
places: the loader guarantees it after an import and the application guarantees it at startup.

### Reading through the model


In [6]:
cheap = await Product.find(Product.kind == "laptop", Product.price < 50).to_list()
print("cheap laptops:", len(cheap))

some = await Product.find(In(Product.maker, ["Aster", "Dalgo"]),
                          Product.stock > 400).sort(Product.price).limit(3).to_list()
for product in some:
    print(f"  {product.sku}  {product.price:8.2f}  {product.maker:7} stock {product.stock}")


cheap laptops: 919
  SKU-0085511      5.12  Dalgo   stock 474
  SKU-0019095      5.12  Dalgo   stock 466
  SKU-0165540      5.19  Aster   stock 497


### A report

The aggregation pipeline, with its output validated into a class:


In [7]:
class KindSummary(pydantic.BaseModel):
    kind: str
    products: int
    stock: int
    average: float


PIPELINE = [
    {"$group": {"_id": "$kind",
                "products": {"$sum": 1},
                "stock": {"$sum": "$stock"},
                "average": {"$avg": "$price"}}},
    {"$project": {"_id": 0, "kind": "$_id", "products": 1, "stock": 1,
                  "average": {"$round": ["$average", 2]}}},
    {"$sort": {"kind": 1}},
]

for row in await Product.aggregate(PIPELINE, projection_model=KindSummary).to_list():
    print(f"  {row.kind:9} {row.products:6} products  {row.stock:8} in stock  "
          f"average {row.average:8.2f}")


  cable      40000 products   9973079 in stock  average  1003.14
  keyboard   40000 products   9943527 in stock  average  1004.66
  laptop     40000 products   9985245 in stock  average  1004.53
  monitor    40000 products  10004592 in stock  average  1001.18
  mouse      40000 products   9977465 in stock  average  1007.40


The `$project` renames `_id` to `kind` because the model has a field of that name, and the names have
to match exactly. That contract is written twice, once in the pipeline and once in the class, and
Pydantic is what checks it.

### When to reach for which

| The job | The tool | Why |
|---|---|---|
| loading many documents | PyMongo `insert_many`, in batches | nothing to validate, nothing to construct |
| a bulk change | PyMongo `update_many` or a pipeline | one operation on the server |
| creating indexes | either, and both is fine | creating one that exists does nothing |
| serving a request | Beanie models | typed attributes, and validation at the edge |
| a filter in application code | Beanie expressions | a typo is an `AttributeError` |
| a report | `aggregate` with a `projection_model` | a typed row rather than a dictionary |
| a change to documents already written | a Beanie migration | it records what it has run |
| anything atomic across two documents | a transaction, on the replica set | one document is free, two are not |

The default is PyMongo for anything that touches a lot of documents at once and Beanie for anything
that serves a request. Mixing them is not a compromise: it is the point, because a collection is not
owned by a library.

### The catalog as one object, finished


In [8]:
class Catalog:
    """The loader is synchronous and the reader is not, which is the shape this guide argues for."""

    def __init__(self, uri=URI, name=CATALOG):
        self.uri, self.name = uri, name
        self.async_client = None

    def load(self, rows, batch=10_000):
        """PyMongo: plain dictionaries, in batches, with the indexes created afterwards."""
        with pymongo.MongoClient(self.uri) as client:
            collection = client.get_default_database()[self.name]
            collection.drop()
            for first in range(0, len(rows), batch):
                collection.insert_many(rows[first:first + batch])
            collection.create_index("sku", name="sku_unique", unique=True)
            collection.create_index([("kind", 1), ("price", 1)], name="kind_price")
            return collection.count_documents({})

    async def start(self):
        """Beanie: one client, opened once, with the models bound to it."""
        self.async_client = AsyncMongoClient(self.uri)
        await init_beanie(database=self.async_client.get_default_database(),
                          document_models=[Product])

    async def stop(self):
        await self.async_client.close()

    async def page(self, kind, under, size=3):
        rows = await (Product.find(Product.kind == kind, Product.price < under)
                             .sort(Product.price).limit(size).to_list())
        return [(product.sku, product.price) for product in rows]

    async def summary(self):
        return await Product.aggregate(PIPELINE, projection_model=KindSummary).to_list()


catalog = Catalog()
print("loaded:", catalog.load(catalog_rows(20_000)), "documents")

await catalog.start()
print("cheapest cables under 50:", await catalog.page("cable", 50))
print("kinds:", [(row.kind, row.products) for row in await catalog.summary()])

await catalog.stop()
await init_beanie(database=async_client.get_default_database(),   # the Catalog closed its own
                  document_models=[Product])                     # client, so bind back to ours


loaded: 20000 documents
cheapest cables under 50: [('SKU-0010509', 5.1), ('SKU-0013599', 6.31), ('SKU-0003949', 6.37)]
kinds: [('cable', 4000), ('keyboard', 4000), ('laptop', 4000), ('monitor', 4000), ('mouse', 4000)]


`load` opens a synchronous client, writes, creates the indexes and closes. `start` opens one
asynchronous client and keeps it, which is where **Connection Pools** applies: a client is opened
once for the life of the program, not per request.

The indexes are created after the load rather than before, which is the faster order: building an
index once over a finished collection beats maintaining it through two hundred thousand inserts.

### Where each part came from

| In the catalog | What it relies on | The notebook that showed it |
|---|---|---|
| `insert_many` in batches | one round trip per batch, under the message limit | **Collections and Documents** |
| `create_index` after the load | an index built once rather than maintained | **Indexes** |
| `unique=True` on `sku` | a constraint that is also the fast path | **Indexes** |
| `(kind, price)` serving two queries | the prefix rule | **Indexes** |
| `Settings.name = CATALOG` | a model over a collection somebody else wrote | **Beanie Documents** |
| `Product.kind == kind` | a filter checked against the model | **Async Queries** |
| `projection_model=KindSummary` | a typed row out of a pipeline | **Async Queries** |
| one `AsyncMongoClient` kept open | a client is not per request | **Beanie Documents** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/16-a-product-catalog-solutions.ipynb).

**1.** Load ten thousand rows with PyMongo and count them.


In [9]:
# your code here


**2.** Explain a query on `maker` before and after creating an index for it.


In [10]:
# your code here


**3.** Read one of the loaded documents back as a Beanie model.


In [11]:
# your code here


**4.** Find the three cheapest products of one kind, through the model.


In [12]:
# your code here


**5.** Produce a per-maker report with a projection model.


In [13]:
# your code here


**6.** Show that creating an index that already exists does nothing.


In [14]:
# your code here


## Common errors

### pydantic_core.ValidationError: the row the loader let through


In [15]:
shop[CATALOG].insert_one({"sku": "SKU-BAD", "name": "no price", "maker": "Aster",
                          "kind": "laptop"})                        # no price, no stock

try:
    await Product.find(Product.sku == "SKU-BAD").to_list()
except pydantic.ValidationError as error:
    problem = error.errors()[0]
    print("ValidationError:", ".".join(str(part) for part in problem["loc"]), "|", problem["msg"])

shop[CATALOG].delete_one({"sku": "SKU-BAD"})
print("removed it again:", shop[CATALOG].count_documents({"sku": "SKU-BAD"}))


ValidationError: price | Field required
removed it again: 0


This is the cost of the split, and it is worth stating plainly rather than pretending it away. The
loader writes dictionaries, so nothing checked them; the reader validates, so the check lands on
whoever reads next.

Two defenses, and they are different jobs. Validate in the loader when the data comes from outside:


In [16]:
def load_validated(rows):
    """Build every model first, so a bad row stops the batch before anything is written."""
    products = [Product(**row) for row in rows]                     # raises here, not later
    return len(products)


try:
    load_validated([{"sku": "S-1", "name": "fine", "maker": "Aster", "kind": "laptop",
                     "price": 1.0},
                    {"sku": "S-2", "name": "bad", "maker": "Aster", "kind": "laptop"}])
except pydantic.ValidationError as error:
    print("the batch never started:", error.errors()[0]["loc"], error.errors()[0]["msg"])


the batch never started: ('price',) Field required


Or put a `$jsonSchema` validator on the collection, so the database refuses what the model would:


In [17]:
shop.command({
    "collMod": CATALOG,
    "validator": {"$jsonSchema": {"bsonType": "object",
                                  "required": ["sku", "name", "maker", "kind", "price"]}},
    "validationLevel": "moderate",
})

try:
    shop[CATALOG].insert_one({"sku": "SKU-BAD2", "name": "still no price"})
except pymongo.errors.WriteError as error:
    print("the database refused it:", failed(error))


the database refused it: WriteError: Document failed validation


Now neither half can write a document the other cannot read. Beanie does not add this for you, which
is why it is worth knowing it exists.

### pymongo.errors.DuplicateKeyError: the unique index doing its job


In [18]:
existing = shop[CATALOG].find_one({}, {"sku": 1})
shop[CATALOG].insert_one({"sku": existing["sku"], "name": "a copy", "maker": "Aster",
                          "kind": "laptop", "price": 1.0})


DuplicateKeyError: E11000 duplicate key error collection: shop.catalog index: sku_unique dup key: { sku: "SKU-0000000" }, full error: {'index': 0, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: shop.catalog index: sku_unique dup key: { sku: "SKU-0000000" }', 'keyPattern': {'sku': 1}, 'keyValue': {'sku': 'SKU-0000000'}}

The index created during the load is a constraint as well as a fast path, so a second product with
the same sku cannot exist however it is written: through PyMongo, through Beanie, or by hand in a
shell.

That is the argument for putting the constraint in the database rather than in the application: the
application is not the only thing that writes.

### No error: the index nobody uses


In [19]:
shop[CATALOG].create_index("name", name="name_1")
print("indexes:", [index["name"] for index in shop[CATALOG].list_indexes()])

print()
print("a query the catalog actually serves:")
print("  ", plan(shop[CATALOG].find({"kind": "laptop", "price": {"$lt": 100}})))
print("an index on name was not used, and is written to on every insert")

shop[CATALOG].drop_index("name_1")
print("dropped it:", [index["name"] for index in shop[CATALOG].list_indexes()])


indexes: ['_id_', 'sku_unique', 'kind_price', 'name_1']

a query the catalog actually serves:
   {'stage': 'IXSCAN', 'documents': 191, 'returned': 191}
an index on name was not used, and is written to on every insert
dropped it: ['_id_', 'sku_unique', 'kind_price']


Nothing failed and nothing warned. An unused index costs disk, costs a write on every insert and
update, and makes the load slower, in exchange for nothing at all.

Indexes come from the queries. An index added because a field "seems important" is a cost with no
matching benefit, and `explain` on the queries you actually serve is how you tell the difference.

### No error: the work a model does on every read


In [20]:
as_dicts = list(shop[CATALOG].find({"kind": "laptop"}))
as_models = await Product.find(Product.kind == "laptop").to_list()

print("the same documents:", len(as_dicts) == len(as_models), f"({len(as_dicts)} of them)")
print("PyMongo gave back:", type(as_dicts[0]).__name__)
print("Beanie gave back: ", type(as_models[0]).__name__)
print()
print("the second one built", len(as_models), "objects and checked every field of every one")


the same documents: True (4000 of them)
PyMongo gave back: dict
Beanie gave back:  Product

the second one built 4000 objects and checked every field of every one


Both calls are correct and neither is fast or slow in the abstract. The difference is where the work
goes: Beanie constructed an object per document and validated six fields on each, and PyMongo handed
back dictionaries straight off the wire.

That validation is the whole reason to use a model, and it is worth paying for on the path that
serves a request, where the data might be anything and a wrong type should be caught. It is worth
nothing on a bulk load of data you generated yourself, which is why the loader in this notebook
never touches the model.

The rule that follows: put the validation where the uncertainty is. For data from outside that is
the loader, not whichever request happens to read the row first.


In [21]:
shop.command({"collMod": CATALOG, "validator": {}, "validationLevel": "off"})
shop[CATALOG].drop()
client.close()
await async_client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- One collection, two libraries, nothing converted between them. PyMongo writes dictionaries and
  Beanie reads models, because BSON is what both of them speak.
- Load with `insert_many` in batches. A batch is sized by the request, not the document, and a
  batch has to fit in one message.
- Create indexes **after** a bulk load rather than before: building one over a finished collection
  beats maintaining it through every insert.
- Indexes come from the queries. `explain`, then `create_index`, and a compound index serves every
  prefix of itself.
- A unique index is a constraint and a fast path at once, and it holds against every writer rather
  than just the one you wrote.
- A model over a collection somebody else wrote validates on read, so a row the loader let through
  becomes the reader's `ValidationError`. Validate in the loader for outside data, or add a
  `$jsonSchema` validator so the database refuses it.
- An unused index costs a write on every insert and returns nothing.
- Reading through a model builds and validates an object per document. That is the point on a
  request path and waste on a bulk load, so put the validation where the uncertainty is.


## What is next

That is the guide. You started with one order written as a document instead of five tables, and you
have finished with a catalog of two hundred thousand products loaded by one library, indexed from
its own query plans, and served through typed models by another.

The natural next step is the asyncpg and psycopg3 guide, which asks the same questions of a
relational server: what an index costs, where a transaction is needed, and what happens when two
writers meet.


---

&#8592; **Previous:** [Migrations](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/15-migrations.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
